# PitchLens — human review of the annotations

This notebook is only used to make the human decisions needed to build `v002`. It never modifies `raw` or `v001`. Every click is saved to `data/processed/v001/qa_review_decisions.csv`, so you can close the notebook and resume later.

## Suggested order

The queue now covers **all 7,152 Infactory source images**, not just the 838 images initially flagged. Decisions already recorded are kept. SoccerEye remains a separate dataset and is not included in this interface.

1. finish the **1,069 test images**;
2. finish the **1,080 validation images**;
3. then review the **5,003 train images**;
4. generate `v002` only when the necessary decisions are done.

The default filter shows all images, in the order test, validation, train, then by sequence and frame number. The absence of an automatic signal does not mean the annotation is correct.

## Review actions

| Button | Future effect in `v002` | When to use it |
|---|---|---|
| **Confirm and advance** | keeps the displayed annotation, including a correction already saved | the annotation is correct |
| **No ball** | writes an empty label | no ball is visible |
| **Keep only this box** | removes the other red boxes and keeps the chosen one | image with two or more boxes of which only one is correct |
| **Save the drawn box** | removes all the old boxes and replaces them with the drawn box | ball visible but box missing, wrong or imprecise |
| **Exclude the image** | removes the image from v002 | no decision possible: occlusion, ambiguity or unusable image |

The red rectangles are the original annotations. For a duplicate, pick the right box in the list, then click **Keep only this box**. To redraw: zoom first if needed, then click **✏️ Enable drawing**; the red boxes then disappear from the central image. Draw the new rectangle, then save it. The button is required because zooming and drawing use the same mouse gesture. The previous and next images give temporal context but are not modified.

In [ ]:
%matplotlib widget

import importlib
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pitchlens.data.review as review_module

# Reload the module if this cell is re-run after an interface improvement.
review_module = importlib.reload(review_module)
ReviewStore = review_module.ReviewStore
ReviewWidget = review_module.ReviewWidget

PROCESSED_ROOT = PROJECT_ROOT / "data/processed/v001"
store = ReviewStore(PROCESSED_ROOT)
store.progress()

## Review by sequence

**First / Last** jump straight to the ends of the sequence. The **Image** slider lets you go directly to any image: drag it or click its value to type a number (position from 1 to N in the sequence).

**Exclude the whole sequence** drops all its images from the future v002, including those already corrected. The previous decisions are saved in `data/processed/v001/sequence_backups/`. **Cancel whole-sequence exclusion** restores them; images that had no decision go back to being pending. Individual decisions changed after the exclusion are kept.

Choose a sequence from the list: the number of images still to review is shown. If it is too degraded, **Skip the rest** (N remaining) excludes all its images without a decision from the future v002, keeping your existing corrections and validations. This includes images you skipped earlier in that sequence. The interface then moves on to the next image without a decision.

To undo this choice, select the sequence again and click **Restore the skipped rest**. Only untouched grouped exclusions go back to pending. The counters distinguish individual decisions from per-sequence exclusions.

The review resumes at the first image without a decision. Images keep a stable order: **Previous** goes back to an image even if already confirmed, **Next** advances without saving. **Next to review** jumps to the remaining decisions.

Three main actions: **Confirm and advance**, **No ball**, **Exclude the image**. Each action saves and advances; you can go back and replace your decision. **Fix the boxes** opens the drawing and selection tools. Notes and origin are in collapsible panels.

In [ ]:
# Close the old interface before recreating it after an update.
previous_review = globals().get("review")
if previous_review is not None:
    previous_review.selector.disconnect_events()
    previous_review.plt.close(previous_review.figure)
    previous_review.ui.close()
review = ReviewWidget(store)
review.display()

## Check progress

This cell re-reads the counters. The decisions file is local and ignored by Git, because it belongs with the dataset rather than the code. It will be used later by the reproducible generation of `v002`.

In [ ]:
store.progress()